<a href="https://colab.research.google.com/github/souha24-ctrl/retail-demand-simulator/blob/main/retail_demand_simulator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Retail Demand & Resource Allocation Simulator

## Research question

Can historical retail transactions support short-term sales forecasting
and help identify potential stock shortages under alternative demand scenarios?

## Project objectives

- Inspect and clean real transactional data.
- Construct daily product-level sales series.
- Compare a machine-learning model with a simple forecasting baseline.
- Simulate demand changes and inventory availability.
- Present forecasts and potential shortages in an interactive dashboard.

## Dataset

UCI Online Retail II.

Source:
https://archive.ics.uci.edu/dataset/502/online+retail+ii

Citation:
Chen, D. (2012). Online Retail II [Dataset].
UCI Machine Learning Repository.
https://doi.org/10.24432/C5CG6D

## Important assumptions

- Recorded sales are a proxy for demand, not a complete measurement of demand.
- Historical stock levels and supplier lead times are not available.
- Inventory and replenishment values are user-defined scenario assumptions.
- Scenario sliders do not represent estimated causal effects.

## Evaluation plan

Use chronological training, validation, and test periods.
Construct features using only past information.
Compare model predictions against a simple baseline before drawing conclusions.

In [ ]:
%pip install -q pandas numpy scikit-learn matplotlib seaborn openpyxl requests joblib

In [ ]:
import sys
import platform

import pandas as pd
import numpy as np
import sklearn
import matplotlib
import seaborn as sns
import openpyxl
import requests
import joblib

environment_versions = {
    "Python": platform.python_version(),
    "pandas": pd.__version__,
    "NumPy": np.__version__,
    "scikit-learn": sklearn.__version__,
    "matplotlib": matplotlib.__version__,
    "seaborn": sns.__version__,
    "openpyxl": openpyxl.__version__,
    "requests": requests.__version__,
    "joblib": joblib.__version__,
}

for package, version in environment_versions.items():
    print(f"{package}: {version}")

print("\nPython executable:", sys.executable)

In [ ]:
from pathlib import Path

PROJECT_DIR = Path("/content/retail-demand-simulator")

folders = [
    "data/raw",
    "data/processed",
    "artifacts",
    "reports/figures",
]

for folder in folders:
    path = PROJECT_DIR / folder
    path.mkdir(parents=True, exist_ok=True)

print("Project working directory:", PROJECT_DIR)

for path in sorted(PROJECT_DIR.rglob("*")):
    if path.is_dir():
        print(path.relative_to(PROJECT_DIR))

## Data source and acquisition

We use the original UCI Online Retail II dataset rather than a modified Kaggle copy.
The dataset contains transaction-level records from a UK-based online retailer between
1 December 2009 and 9 December 2011.

The raw file is downloaded into the Colab working directory and is not modified.
All cleaning and feature engineering will create separate processed files.

Source:
https://archive.ics.uci.edu/dataset/502/online+retail+ii

Citation:
Chen, D. (2012). Online Retail II [Dataset].
UCI Machine Learning Repository.
https://doi.org/10.24432/C5CG6D

In [ ]:
import requests
from pathlib import Path

RAW_DIR = PROJECT_DIR / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

DATASET_URL = "https://archive.ics.uci.edu/static/public/502/online+retail+ii.zip"
ZIP_PATH = RAW_DIR / "online_retail_ii.zip"

response = requests.get(DATASET_URL, timeout=120)
response.raise_for_status()

ZIP_PATH.write_bytes(response.content)

print("Downloaded:", ZIP_PATH)
print("Size (MB):", round(ZIP_PATH.stat().st_size / 1024 / 1024, 2))

In [ ]:
import zipfile

with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
    print("Files inside ZIP:")
    for name in zip_ref.namelist():
        print("-", name)

    zip_ref.extractall(RAW_DIR)

xlsx_files = list(RAW_DIR.glob("*.xlsx"))
print("\nExcel files found:", xlsx_files)

In [ ]:
import pandas as pd

XLSX_PATH = next(RAW_DIR.glob("*.xlsx"))

excel_sheets = pd.ExcelFile(XLSX_PATH).sheet_names

print("Excel file:", XLSX_PATH.name)
print("Sheets:", excel_sheets)

for sheet in excel_sheets:
    preview = pd.read_excel(
        XLSX_PATH,
        sheet_name=sheet,
        nrows=5
    )
    print(f"\n--- Sheet: {sheet} ---")
    print(preview)

In [ ]:
# Standard column names used throughout the project
COLUMN_MAPPING = {
    "Invoice": "invoice_no",
    "StockCode": "stock_code",
    "Description": "description",
    "Quantity": "quantity",
    "InvoiceDate": "invoice_date",
    "Price": "unit_price",
    "Customer ID": "customer_id",
    "Country": "country",
}

frames = []

for sheet in excel_sheets:
    sheet_df = pd.read_excel(XLSX_PATH, sheet_name=sheet)

    # Standardize column names
    sheet_df = sheet_df.rename(columns=COLUMN_MAPPING)

    # Add provenance metadata
    sheet_df["source_sheet"] = sheet

    frames.append(sheet_df)
    print(f"Loaded {sheet}: {len(sheet_df):,} rows")

raw_df = pd.concat(frames, ignore_index=True)

# Convert identifiers to strings to preserve cancellation prefixes such as C489449
raw_df["invoice_no"] = raw_df["invoice_no"].astype(str)
raw_df["stock_code"] = raw_df["stock_code"].astype(str)

# Convert date and numeric fields explicitly
raw_df["invoice_date"] = pd.to_datetime(raw_df["invoice_date"], errors="coerce")
raw_df["quantity"] = pd.to_numeric(raw_df["quantity"], errors="coerce")
raw_df["unit_price"] = pd.to_numeric(raw_df["unit_price"], errors="coerce")

print("\nCombined raw dataset shape:", raw_df.shape)
print("\nColumns:")
print(raw_df.columns.tolist())

raw_df.head()

In [ ]:
# Identify values in description that are not already text
non_string_description_mask = raw_df["description"].apply(
    lambda value: not isinstance(value, str)
)

print("Non-string description values:", int(non_string_description_mask.sum()))
print("Percentage:", f"{100 * non_string_description_mask.mean():.4f}%")

raw_df.loc[
    non_string_description_mask,
    ["invoice_no", "stock_code", "description", "quantity", "invoice_date"]
].head(20)

In [ ]:
# Convert identifiers to strings to preserve cancellation prefixes such as C489449
raw_df["invoice_no"] = raw_df["invoice_no"].astype("string")
raw_df["stock_code"] = raw_df["stock_code"].astype("string")

# Standardize free-text fields using nullable string dtype
raw_df["description"] = raw_df["description"].astype("string")
raw_df["country"] = raw_df["country"].astype("string")

# Convert date and numeric fields explicitly
raw_df["invoice_date"] = pd.to_datetime(raw_df["invoice_date"], errors="coerce")
raw_df["quantity"] = pd.to_numeric(raw_df["quantity"], errors="coerce")
raw_df["unit_price"] = pd.to_numeric(raw_df["unit_price"], errors="coerce")

print("\nCombined raw dataset shape:", raw_df.shape)
print("\nColumn data types:")
print(raw_df.dtypes)

raw_df.head()

In [ ]:
RAW_PARQUET_PATH = PROJECT_DIR / "data" / "raw" / "online_retail_ii_raw.parquet"

raw_df.to_parquet(RAW_PARQUET_PATH, index=False)

print("Saved raw snapshot:", RAW_PARQUET_PATH)
print("Snapshot shape:", raw_df.shape)

print("\nColumn data types:")
print(raw_df.dtypes)

### Data-type issue: mixed description values

The original `Description` column contained both text and non-text values.
Saving the raw DataFrame directly to Parquet failed because PyArrow could not infer
a single column type from mixed Python objects.

We standardized identifier and text columns using pandas nullable string types before saving.
This preserves cancellation prefixes in invoice numbers and retains missing descriptions as
missing values rather than converting them into the literal text `"nan"`.

In [ ]:
# Recreate audit masks so this cell does not depend on earlier notebook execution
cancellation_mask = raw_df["invoice_no"].str.upper().str.startswith("C")
negative_quantity_mask = raw_df["quantity"] < 0
zero_or_negative_price_mask = raw_df["unit_price"] <= 0

audit_report = pd.DataFrame([
    {"metric": "total_rows", "value": len(raw_df)},
    {"metric": "total_columns", "value": raw_df.shape[1]},
    {"metric": "exact_duplicate_rows", "value": int(raw_df.duplicated().sum())},
    {"metric": "cancellation_rows", "value": int(cancellation_mask.sum())},
    {"metric": "negative_quantity_rows", "value": int(negative_quantity_mask.sum())},
    {"metric": "zero_or_negative_price_rows", "value": int(zero_or_negative_price_mask.sum())},
    {"metric": "missing_customer_id_rows", "value": int(raw_df["customer_id"].isna().sum())},
    {"metric": "missing_description_rows", "value": int(raw_df["description"].isna().sum())},
    {"metric": "invalid_dates", "value": int(raw_df["invoice_date"].isna().sum())},
    {"metric": "minimum_date", "value": str(raw_df["invoice_date"].min())},
    {"metric": "maximum_date", "value": str(raw_df["invoice_date"].max())},
    {"metric": "unique_products", "value": int(raw_df["stock_code"].nunique())},
])

AUDIT_PATH = PROJECT_DIR / "reports" / "data_quality_audit.csv"
audit_report.to_csv(AUDIT_PATH, index=False)

print("Saved audit report:", AUDIT_PATH)
audit_report

### Notebook execution note

The audit masks are recreated within the audit-report cell so that the report remains
reproducible even if earlier audit cells are modified or rerun in a different order.

This is important in notebooks because variables exist in the execution environment,
not in the saved notebook file itself.

### Initial data-quality interpretation

The raw dataset contains more than one million transaction lines across two annual Excel sheets.
Missing descriptions affect less than 0.5% of rows, while missing customer identifiers are more
common because some transactions may be anonymous or incomplete.

Cancellations, negative quantities, and non-positive prices are recorded separately rather than
automatically removed. They may represent returns, corrections, or data-entry issues, and their
treatment must be justified in the cleaning pipeline.

## Cleaning rules for demand forecasting

The goal is to estimate daily product-level sales volume. We apply the following rules:

1. Remove exact duplicate transaction rows.
2. Remove cancellation invoices, identified by invoice numbers beginning with `C`.
3. Remove rows with negative quantities, which are associated with returns or corrections.
4. Remove rows with zero or negative unit prices, which are not valid retail sales.
5. Retain rows with missing customer IDs because customer identity is not required for
   product-level demand forecasting.
6. Retain missing descriptions because the product code is the forecasting key.
7. Aggregate valid transactions to daily sales quantity for each product.

These rules are applied only after the raw audit. The raw dataset remains unchanged.

In [ ]:
# Start from the untouched raw snapshot
cleaning_df = raw_df.copy()

initial_rows = len(cleaning_df)

# 1. Remove exact duplicate transaction rows
cleaning_df = cleaning_df.drop_duplicates()
after_duplicates = len(cleaning_df)

# 2. Remove cancellation invoices
cancellation_mask = cleaning_df["invoice_no"].str.upper().str.startswith("C")
cleaning_df = cleaning_df[~cancellation_mask]
after_cancellations = len(cleaning_df)

# 3. Remove negative quantities
negative_quantity_mask = cleaning_df["quantity"] < 0
cleaning_df = cleaning_df[~negative_quantity_mask]
after_negative_quantity = len(cleaning_df)

# 4. Remove zero or negative unit prices
invalid_price_mask = cleaning_df["unit_price"] <= 0
cleaning_df = cleaning_df[~invalid_price_mask]
after_invalid_price = len(cleaning_df)

# Create a cleaning summary
cleaning_summary = pd.DataFrame([
    {"step": "Raw rows", "rows_remaining": initial_rows},
    {"step": "After removing exact duplicates", "rows_remaining": after_duplicates},
    {"step": "After removing cancellations", "rows_remaining": after_cancellations},
    {"step": "After removing negative quantities", "rows_remaining": after_negative_quantity},
    {"step": "After removing invalid prices", "rows_remaining": after_invalid_price},
])

cleaning_summary["rows_removed"] = cleaning_summary["rows_remaining"].shift(1) - cleaning_summary["rows_remaining"]
cleaning_summary.loc[0, "rows_removed"] = 0

cleaning_summary

### Cleaning interpretation

Exact duplicates, cancellations, negative quantities, and invalid prices are removed because
they do not represent ordinary positive sales transactions.

Missing customer IDs and descriptions are retained because they do not prevent aggregation of
product-level daily demand. This distinction avoids unnecessarily discarding valid sales records.

In [ ]:
# Create a date column without time
cleaning_df["sale_date"] = cleaning_df["invoice_date"].dt.normalize()

# Aggregate valid sales to daily product-level demand
daily_demand_df = (
    cleaning_df
    .groupby(["stock_code", "sale_date"], as_index=False)
    .agg(
        daily_quantity=("quantity", "sum"),
        transaction_count=("invoice_no", "nunique"),
        average_unit_price=("unit_price", "mean"),
    )
    .sort_values(["stock_code", "sale_date"])
)

print("Daily demand dataset shape:", daily_demand_df.shape)
print("Unique products:", daily_demand_df["stock_code"].nunique())
print("Date range:", daily_demand_df["sale_date"].min(), "to", daily_demand_df["sale_date"].max())

daily_demand_df.head()

In [ ]:
# Product eligibility criteria
MIN_HISTORY_DAYS = 180
MIN_TOTAL_QUANTITY = 100
MIN_ACTIVE_DAYS = 30

product_activity = (
    daily_demand_df
    .groupby("stock_code")
    .agg(
        history_days=("sale_date", "nunique"),
        first_sale=("sale_date", "min"),
        last_sale=("sale_date", "max"),
        total_quantity=("daily_quantity", "sum"),
    )
)

product_activity["calendar_span_days"] = (
    product_activity["last_sale"] - product_activity["first_sale"]
).dt.days + 1

eligible_products = product_activity[
    (product_activity["history_days"] >= MIN_ACTIVE_DAYS)
    & (product_activity["total_quantity"] >= MIN_TOTAL_QUANTITY)
    & (product_activity["calendar_span_days"] >= MIN_HISTORY_DAYS)
].copy()

eligible_products = eligible_products.sort_values(
    by="total_quantity",
    ascending=False
)

print("Eligible products:", len(eligible_products))
eligible_products.head(20)

In [ ]:
TOP_N_PRODUCTS = 20

top_products = eligible_products.head(TOP_N_PRODUCTS).index.tolist()

forecast_df = daily_demand_df[
    daily_demand_df["stock_code"].isin(top_products)
].copy()

print("Selected products:", len(top_products))
print("Forecast dataset shape:", forecast_df.shape)
forecast_df.head()

### Product-selection rationale

The first forecasting version focuses on the 20 products with the highest total observed sales
volume among products with sufficient history and activity.

This scope keeps model training, evaluation, and dashboard interaction manageable. It also avoids
forecasting sparse products with too few historical observations.

This is a scope decision, not a claim that these are the only products relevant to allocation.

In [ ]:
# Create a complete daily date range for the forecasting period
full_date_range = pd.date_range(
    start=forecast_df["sale_date"].min(),
    end=forecast_df["sale_date"].max(),
    freq="D"
)

# Create every product-date combination
product_index = pd.MultiIndex.from_product(
    [top_products, full_date_range],
    names=["stock_code", "sale_date"]
)

complete_daily_df = (
    forecast_df
    .set_index(["stock_code", "sale_date"])
    .reindex(product_index)
    .reset_index()
)

# Days with no recorded sale are treated as zero observed sales
complete_daily_df["daily_quantity"] = complete_daily_df["daily_quantity"].fillna(0)
complete_daily_df["transaction_count"] = complete_daily_df["transaction_count"].fillna(0)

# Fill price using each product's most recent known price
complete_daily_df["average_unit_price"] = (
    complete_daily_df
    .groupby("stock_code")["average_unit_price"]
    .transform(lambda series: series.ffill().bfill())
)

print("Complete daily dataset shape:", complete_daily_df.shape)
complete_daily_df.head()

### Zero-sales assumption

Days without a recorded transaction are treated as zero observed sales.

This is a practical forecasting assumption, but it is not equivalent to zero demand: a product
may have been unavailable, out of stock, or simply not purchased. Because historical stock levels
are unavailable, this limitation cannot be resolved from the transaction data alone.

In [ ]:
PROCESSED_PATH = PROJECT_DIR / "data" / "processed" / "daily_product_demand_top20.parquet"

complete_daily_df.to_parquet(PROCESSED_PATH, index=False)

print("Saved processed dataset:", PROCESSED_PATH)
print("Shape:", complete_daily_df.shape)
print("Products:", complete_daily_df["stock_code"].nunique())
print("Date range:", complete_daily_df["sale_date"].min(), "to", complete_daily_df["sale_date"].max())

In [ ]:
import matplotlib.pyplot as plt

daily_total_demand = (
    complete_daily_df
    .groupby("sale_date")["daily_quantity"]
    .sum()
)

plt.figure(figsize=(14, 5))
plt.plot(daily_total_demand.index, daily_total_demand.values)
plt.title("Total daily sales quantity: top 20 products")
plt.xlabel("Date")
plt.ylabel("Units sold")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Exploratory interpretation

The total daily demand series shows substantial variation over time. Retail demand often contains
weekly seasonality, holiday effects, and irregular spikes.

The forecasting model will use calendar features and lagged sales information. All lagged features
will use only past observations, so the model cannot use future sales to predict earlier dates.

## Investigation of unusually large sales days

The exploratory daily-sales chart contains a small number of unusually large spikes.
Before training, we inspect the underlying transactions.

A large quantity is not automatically an error. It may represent a genuine bulk order,
a correction, or a transaction that was subsequently cancelled.

No outlier is removed solely because it is large.

In [ ]:
largest_days = (
    complete_daily_df
    .groupby("sale_date", as_index=False)["daily_quantity"]
    .sum()
    .sort_values("daily_quantity", ascending=False)
    .head(10)
)

print("Ten largest sales days for the exploratory top-20 selection:")
display(largest_days)

spike_date = largest_days.iloc[0]["sale_date"]

spike_products = (
    daily_demand_df.loc[
        daily_demand_df["sale_date"].eq(spike_date)
        & daily_demand_df["stock_code"].isin(top_products)
    ]
    .sort_values("daily_quantity", ascending=False)
)

print("\nProduct contributions on the largest day:", spike_date.date())
display(spike_products)

In [ ]:
spike_stock_code = spike_products.iloc[0]["stock_code"]

spike_transactions = (
    cleaning_df.loc[
        cleaning_df["invoice_date"].dt.normalize().eq(spike_date)
        & cleaning_df["stock_code"].eq(spike_stock_code)
    ]
    .sort_values("quantity", ascending=False)
)

columns_to_show = [
    "invoice_no",
    "stock_code",
    "description",
    "quantity",
    "invoice_date",
    "unit_price",
    "customer_id",
    "country",
]

print("Largest-contributing product:", spike_stock_code)
print("Transactions on the spike date:")
display(spike_transactions[columns_to_show].head(20))

In [ ]:
largest_transaction = spike_transactions.iloc[0]
spike_customer_id = largest_transaction["customer_id"]

window_start = spike_date - pd.Timedelta(days=7)
window_end = spike_date + pd.Timedelta(days=8)

related_mask = (
    raw_df["stock_code"].eq(spike_stock_code)
    & raw_df["invoice_date"].ge(window_start)
    & raw_df["invoice_date"].lt(window_end)
)

if pd.notna(spike_customer_id):
    related_mask &= raw_df["customer_id"].eq(spike_customer_id)

related_transactions = (
    raw_df.loc[related_mask, columns_to_show]
    .sort_values("invoice_date")
)

print("Nearby transactions for the same product")
if pd.notna(spike_customer_id):
    print("Restricted to customer:", spike_customer_id)
else:
    print("Customer ID unavailable; showing all customers for this product.")

display(related_transactions.head(50))

In [ ]:
REPORTS_DIR = PROJECT_DIR / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

largest_days.to_csv(
    REPORTS_DIR / "largest_sales_days.csv",
    index=False
)

spike_transactions.to_csv(
    REPORTS_DIR / "largest_day_transactions.csv",
    index=False
)

related_transactions.to_csv(
    REPORTS_DIR / "spike_related_transactions.csv",
    index=False
)

print("Saved three diagnostic tables in the Colab reports folder.")

### Investigation outcome: a large order reversed on the same day

The largest observed spike was driven by invoice `541431`, which recorded 74,215 units
of product `23166` for customer `12346` at 10:01 on 18 January 2011.

Invoice `C541433` recorded −74,215 units of the same product for the same customer
at the same unit price at 10:17.

We treat this inspected pair as a same-day reversal and exclude the positive order
from the forecasting sales table. Both original records remain in the raw dataset.

This is a documented, case-specific exclusion and not a general rule that large orders
are invalid. Other cancellations and returns have not been comprehensively matched.

In [ ]:
positive_mask = (
    raw_df["invoice_no"].eq("541431")
    & raw_df["stock_code"].eq("23166")
    & raw_df["customer_id"].eq(12346)
    & raw_df["quantity"].eq(74215)
    & raw_df["invoice_date"].eq(pd.Timestamp("2011-01-18 10:01:00"))
)

reversal_mask = (
    raw_df["invoice_no"].eq("C541433")
    & raw_df["stock_code"].eq("23166")
    & raw_df["customer_id"].eq(12346)
    & raw_df["quantity"].eq(-74215)
    & raw_df["invoice_date"].eq(pd.Timestamp("2011-01-18 10:17:00"))
)

positive_rows = raw_df.loc[positive_mask]
reversal_rows = raw_df.loc[reversal_mask]

assert len(positive_rows) == 1, "Expected exactly one positive transaction."
assert len(reversal_rows) == 1, "Expected exactly one cancellation transaction."

assert (
    positive_rows.iloc[0]["unit_price"]
    == reversal_rows.iloc[0]["unit_price"]
), "The unit prices do not match."

print("Verified: one positive transaction and one matching same-day cancellation.")
display(
    raw_df.loc[positive_mask | reversal_mask, columns_to_show]
    .sort_values("invoice_date")
)

In [ ]:
verified_reversal_mask = (
    cleaning_df["invoice_no"].eq("541431")
    & cleaning_df["stock_code"].eq("23166")
    & cleaning_df["customer_id"].eq(12346)
    & cleaning_df["quantity"].eq(74215)
    & cleaning_df["invoice_date"].eq(pd.Timestamp("2011-01-18 10:01:00"))
)

assert verified_reversal_mask.sum() == 1, (
    "Expected exactly one positive row in cleaning_df."
)

sales_df = cleaning_df.loc[~verified_reversal_mask].copy()

print("Rows before case-specific exclusion:", f"{len(cleaning_df):,}")
print("Rows after exclusion:", f"{len(sales_df):,}")
print(
    "Units excluded:",
    f"{cleaning_df.loc[verified_reversal_mask, 'quantity'].sum():,}"
)

assert not sales_df["invoice_no"].eq("541431").any()

In [ ]:
exclusion_audit = pd.DataFrame([{
    "excluded_invoice": "541431",
    "related_cancellation_invoice": "C541433",
    "stock_code": "23166",
    "excluded_quantity": 74215,
    "transaction_date": "2011-01-18",
    "reason": (
        "Inspected same-day reversal: matching customer, product, "
        "quantity magnitude, and unit price; cancellation 16 minutes later."
    ),
}])

exclusion_path = PROJECT_DIR / "reports" / "verified_transaction_exclusions.csv"
exclusion_audit.to_csv(exclusion_path, index=False)

display(exclusion_audit)
print("Saved exclusion audit:", exclusion_path)

In [ ]:
sales_df["sale_date"] = sales_df["invoice_date"].dt.normalize()

corrected_spike_day_total = sales_df.loc[
    sales_df["sale_date"].eq(pd.Timestamp("2011-01-18"))
    & sales_df["stock_code"].isin(top_products),
    "quantity"
].sum()

print(
    "Corrected sales on 2011-01-18 for the exploratory top 20:",
    f"{corrected_spike_day_total:,.0f} units"
)

## Forecasting protocol

The model predicts one day ahead for a fixed set of products selected from training data.

Each prediction uses information available before the target day. During validation and
testing, actual sales from previous days may be used to update lagged features, but the
model must not use sales from the target day or later.

This is rolling one-day-ahead forecasting, not a multi-day forecast issued at one origin.

The target is recorded positive sales quantity after documented exclusions.
It is not net sales after all returns, nor a complete measure of customer demand.

In [ ]:
DATA_START = pd.Timestamp("2009-12-01")
TRAIN_END = pd.Timestamp("2011-07-31")
VALIDATION_END = pd.Timestamp("2011-09-30")
MODEL_END = pd.Timestamp("2011-12-08")

model_sales_df = sales_df.loc[
    sales_df["sale_date"].between(DATA_START, MODEL_END)
].copy()

training_sales_df = model_sales_df.loc[
    model_sales_df["sale_date"] <= TRAIN_END
].copy()

print("Training ends:", TRAIN_END.date())
print("Validation ends:", VALIDATION_END.date())
print("Test ends:", MODEL_END.date())

print("\nRows excluded after the modeling cutoff:")
print(int((sales_df["sale_date"] > MODEL_END).sum()))

In [ ]:
MIN_ACTIVE_DAYS = 30
MIN_CALENDAR_SPAN_DAYS = 180
MIN_TRAINING_UNITS = 100
TOP_N_PRODUCTS = 20

training_product_summary = (
    training_sales_df
    .groupby("stock_code")
    .agg(
        active_sales_days=("sale_date", "nunique"),
        first_observed_sale=("sale_date", "min"),
        last_training_sale=("sale_date", "max"),
        training_units=("quantity", "sum"),
    )
    .reset_index()
)

training_product_summary["calendar_span_days"] = (
    training_product_summary["last_training_sale"]
    - training_product_summary["first_observed_sale"]
).dt.days + 1

eligible_training_products = (
    training_product_summary.loc[
        (training_product_summary["active_sales_days"] >= MIN_ACTIVE_DAYS)
        & (
            training_product_summary["calendar_span_days"]
            >= MIN_CALENDAR_SPAN_DAYS
        )
        & (
            training_product_summary["training_units"]
            >= MIN_TRAINING_UNITS
        )
    ]
    .sort_values(
        ["training_units", "stock_code"],
        ascending=[False, True]
    )
)

selected_product_summary = (
    eligible_training_products
    .head(TOP_N_PRODUCTS)
    .copy()
)

selected_products = selected_product_summary["stock_code"].tolist()

assert len(selected_products) == TOP_N_PRODUCTS

print("Eligible products using training data:", len(eligible_training_products))
print("Selected products:", len(selected_products))

display(selected_product_summary)

In [ ]:
selected_daily_sales = (
    model_sales_df.loc[
        model_sales_df["stock_code"].isin(selected_products)
    ]
    .groupby(["stock_code", "sale_date"], as_index=False)
    .agg(
        daily_quantity=("quantity", "sum")
    )
    .sort_values(["stock_code", "sale_date"])
)

print("Observed product-date rows:", len(selected_daily_sales))
display(selected_daily_sales.head())

In [ ]:
product_calendars = []

for row in selected_product_summary.itertuples(index=False):
    product_calendar = pd.DataFrame({
        "stock_code": row.stock_code,
        "sale_date": pd.date_range(
            start=row.first_observed_sale,
            end=MODEL_END,
            freq="D"
        )
    })

    product_calendars.append(product_calendar)

calendar_df = pd.concat(product_calendars, ignore_index=True)

daily_model_df = calendar_df.merge(
    selected_daily_sales,
    on=["stock_code", "sale_date"],
    how="left",
    validate="one_to_one"
)

daily_model_df["had_sales_record"] = (
    daily_model_df["daily_quantity"].notna()
)

daily_model_df["daily_quantity"] = (
    daily_model_df["daily_quantity"]
    .fillna(0)
    .astype("int64")
)

daily_model_df = (
    daily_model_df
    .sort_values(["stock_code", "sale_date"])
    .reset_index(drop=True)
)

assert not daily_model_df.duplicated(
    ["stock_code", "sale_date"]
).any()

assert daily_model_df["daily_quantity"].ge(0).all()

print("Rebuilt daily table shape:", daily_model_df.shape)
print("Products:", daily_model_df["stock_code"].nunique())

display(daily_model_df.head())

In [ ]:
daily_model_df["split"] = "test"

daily_model_df.loc[
    daily_model_df["sale_date"] <= VALIDATION_END,
    "split"
] = "validation"

daily_model_df.loc[
    daily_model_df["sale_date"] <= TRAIN_END,
    "split"
] = "train"

split_summary = (
    daily_model_df
    .groupby("split")
    .agg(
        rows=("daily_quantity", "size"),
        products=("stock_code", "nunique"),
        start_date=("sale_date", "min"),
        end_date=("sale_date", "max"),
        total_units=("daily_quantity", "sum"),
    )
    .reindex(["train", "validation", "test"])
)

display(split_summary)

In [ ]:
daily_model_df.to_parquet(
    PROJECT_DIR / "data" / "processed" / "daily_sales_modeling.parquet",
    index=False
)

selected_product_summary.to_csv(
    PROJECT_DIR / "reports" / "training_selected_products.csv",
    index=False
)

split_summary.to_csv(
    PROJECT_DIR / "reports" / "chronological_split_summary.csv"
)

print("Saved modeling table, product-selection record, and split summary.")

### Modeling-table assumptions and limitations

Products are selected using training-period transactions only. Their daily calendars begin
at the first observed training-period sale and continue through the modeling cutoff.

Dates without a recorded sale are assigned zero observed sales. This does not establish
zero demand or confirm product availability: stockouts, closures, and discontinuation
cannot be reliably distinguished using these transactions alone.

The final extract date is excluded because it may be incomplete.

No backward-filled prices or same-day transaction-derived predictors are used.
The `had_sales_record` column is retained for auditing only.

### Connection to stochastic processes

For each product, daily recorded sales can be modeled as a discrete-time stochastic
process {Y_t}. The historical sales series is one observed realization of that process.

The forecasting task is to estimate Y_(t+1) using information available through day t.
Under squared-error loss, the theoretically optimal predictor is the conditional
expectation E[Y_(t+1) | F_t], where F_t represents the information available at time t.

The machine-learning model approximates this relationship using past sales, calendar
features, and product identity.

This approach does not assume that sales form a Markov process or establish stationarity.
The initial what-if simulator will apply deterministic scenario changes to point forecasts;
it will not yet simulate random demand trajectories or estimate shortage probabilities.

## Past-only feature engineering and forecasting baselines

Each row represents a product and a target date.

The target is the recorded sales quantity on that date. Predictors include:
- sales 1, 7, and 14 calendar days earlier;
- mean sales over the preceding 7 and 28 days;
- known calendar information;
- product identity.

Rolling averages are shifted before calculation so that they exclude the target day's sales.

We compare two simple baselines on validation data:
1. Previous-week same-day sales.
2. Mean sales over the preceding seven days.

The test period remains reserved for final evaluation.

In [ ]:
feature_df = (
    daily_model_df
    .sort_values(["stock_code", "sale_date"])
    .reset_index(drop=True)
    .copy()
)

assert not feature_df.duplicated(["stock_code", "sale_date"]).any()

day_gaps = (
    feature_df
    .groupby("stock_code")["sale_date"]
    .diff()
    .dropna()
)

assert day_gaps.eq(pd.Timedelta(days=1)).all(), (
    "A product calendar contains missing or repeated dates."
)

print("Calendar checks passed: each product has consecutive daily rows.")

In [ ]:
for lag in [1, 7, 14]:
    feature_df[f"lag_{lag}"] = (
        feature_df
        .groupby("stock_code")["daily_quantity"]
        .shift(lag)
    )

display(
    feature_df[
        ["stock_code", "sale_date", "daily_quantity", "lag_1", "lag_7", "lag_14"]
    ].head(16)
)

In [ ]:
for window in [7, 28]:
    feature_df[f"rolling_mean_{window}"] = (
        feature_df
        .groupby("stock_code")["daily_quantity"]
        .transform(
            lambda series: (
                series
                .shift(1)
                .rolling(window=window, min_periods=window)
                .mean()
            )
        )
    )

display(
    feature_df[
        [
            "stock_code",
            "sale_date",
            "daily_quantity",
            "lag_1",
            "rolling_mean_7",
            "rolling_mean_28",
        ]
    ].head(30)
)

In [ ]:
feature_df["day_of_week"] = feature_df["sale_date"].dt.dayofweek
feature_df["month"] = feature_df["sale_date"].dt.month

display(
    feature_df[
        ["stock_code", "sale_date", "day_of_week", "month"]
    ].head(10)
)

In [ ]:
NUMERIC_FEATURES = [
    "lag_1",
    "lag_7",
    "lag_14",
    "rolling_mean_7",
    "rolling_mean_28",
    "day_of_week",
    "month",
]

FEATURE_COLUMNS = ["stock_code"] + NUMERIC_FEATURES
TARGET_COLUMN = "daily_quantity"

modeling_df = (
    feature_df
    .dropna(subset=NUMERIC_FEATURES)
    .reset_index(drop=True)
)

feature_split_summary = (
    modeling_df
    .groupby("split")
    .agg(
        rows=("daily_quantity", "size"),
        products=("stock_code", "nunique"),
        first_target_date=("sale_date", "min"),
        last_target_date=("sale_date", "max"),
    )
    .reindex(["train", "validation", "test"])
)

print("Rows removed for insufficient history:", len(feature_df) - len(modeling_df))
display(feature_split_summary)

assert modeling_df[FEATURE_COLUMNS].notna().all().all()
assert len(modeling_df.loc[modeling_df["split"].eq("validation")]) == 1220
assert len(modeling_df.loc[modeling_df["split"].eq("test")]) == 1380

In [ ]:
train_df = modeling_df.loc[
    modeling_df["split"].eq("train")
].copy()

validation_df = modeling_df.loc[
    modeling_df["split"].eq("validation")
].copy()

test_df = modeling_df.loc[
    modeling_df["split"].eq("test")
].copy()

X_train = train_df[FEATURE_COLUMNS]
y_train = train_df[TARGET_COLUMN]

X_validation = validation_df[FEATURE_COLUMNS]
y_validation = validation_df[TARGET_COLUMN]

print("Training rows:", len(train_df))
print("Validation rows:", len(validation_df))
print("Reserved test rows:", len(test_df))

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

def regression_metrics(y_true, y_pred):
    actual = np.asarray(y_true, dtype=float)
    predicted = np.asarray(y_pred, dtype=float)

    total_actual = actual.sum()

    return {
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
        "WAPE_percent": (
            100 * np.abs(actual - predicted).sum() / total_actual
            if total_actual > 0
            else np.nan
        ),
        "mean_error_pred_minus_actual": (predicted - actual).mean(),
    }

validation_df["pred_previous_week"] = validation_df["lag_7"]
validation_df["pred_previous_7day_mean"] = validation_df["rolling_mean_7"]

baseline_results = pd.DataFrame([
    {
        "model": "Previous-week same day",
        **regression_metrics(
            validation_df[TARGET_COLUMN],
            validation_df["pred_previous_week"]
        )
    },
    {
        "model": "Previous 7-day mean",
        **regression_metrics(
            validation_df[TARGET_COLUMN],
            validation_df["pred_previous_7day_mean"]
        )
    }
])

display(baseline_results.round(3))

In [ ]:
product_baseline_results = []

for stock_code, product_data in validation_df.groupby("stock_code"):
    metrics = regression_metrics(
        product_data[TARGET_COLUMN],
        product_data["pred_previous_week"]
    )

    product_baseline_results.append({
        "stock_code": stock_code,
        "actual_units": product_data[TARGET_COLUMN].sum(),
        **metrics
    })

product_baseline_results = (
    pd.DataFrame(product_baseline_results)
    .sort_values("MAE", ascending=False)
)

display(product_baseline_results.round(3))

In [ ]:
modeling_df.to_parquet(
    PROJECT_DIR / "data" / "processed" / "daily_sales_features.parquet",
    index=False
)

baseline_results.to_csv(
    PROJECT_DIR / "reports" / "validation_baseline_metrics.csv",
    index=False
)

product_baseline_results.to_csv(
    PROJECT_DIR / "reports" / "validation_previous_week_by_product.csv",
    index=False
)

print("Saved feature table and validation baseline reports.")

## Random Forest: first supervised forecasting experiment

We train a single Random Forest regression model across the selected 20 products.

Inputs:
- Product identity.
- Sales 1, 7, and 14 days earlier.
- Mean sales over the preceding 7 and 28 days.
- Target-day weekday and month.

Product identity is one-hot encoded rather than treated as a numerical measurement.

The model is trained only on the training period. Validation data is used to compare
its forecasts with simple baselines. The test period remains untouched.

The primary validation metric is MAE. RMSE, WAPE, and mean error are reported
as complementary diagnostics.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor

preprocessor = ColumnTransformer(
    transformers=[
        (
            "product",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            ["stock_code"]
        ),
        (
            "numeric",
            "passthrough",
            NUMERIC_FEATURES
        ),
    ],
    remainder="drop"
)

rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    (
        "model",
        RandomForestRegressor(
            n_estimators=200,
            max_depth=12,
            min_samples_leaf=5,
            max_features=1.0,
            random_state=42,
            n_jobs=2
        )
    )
])

rf_pipeline

In [ ]:
from time import perf_counter

assert X_train.columns.tolist() == FEATURE_COLUMNS
assert X_validation.columns.tolist() == FEATURE_COLUMNS

assert train_df["sale_date"].max() < validation_df["sale_date"].min()

training_start = perf_counter()

rf_pipeline.fit(X_train, y_train)

training_seconds = perf_counter() - training_start

print(f"Training completed in {training_seconds:.2f} seconds.")
print("Training rows:", len(X_train))
print("Input columns:", FEATURE_COLUMNS)

In [ ]:
validation_df["pred_random_forest"] = rf_pipeline.predict(
    X_validation
)

display(
    validation_df[
        [
            "stock_code",
            "sale_date",
            "daily_quantity",
            "pred_previous_week",
            "pred_previous_7day_mean",
            "pred_random_forest",
        ]
    ].head(15)
)

In [ ]:
validation_df["pred_zero"] = 0.0

prediction_columns = {
    "Always zero reference": "pred_zero",
    "Previous-week same day": "pred_previous_week",
    "Previous 7-day mean": "pred_previous_7day_mean",
    "Random Forest v1": "pred_random_forest",
}

validation_comparison = pd.DataFrame([
    {
        "model": model_name,
        **regression_metrics(
            validation_df[TARGET_COLUMN],
            validation_df[prediction_column]
        )
    }
    for model_name, prediction_column in prediction_columns.items()
]).sort_values("MAE").reset_index(drop=True)

display(validation_comparison.round(3))

mean_baseline_mae = mean_absolute_error(
    y_validation,
    validation_df["pred_previous_7day_mean"]
)

rf_mae = mean_absolute_error(
    y_validation,
    validation_df["pred_random_forest"]
)

improvement_percent = (
    100 * (mean_baseline_mae - rf_mae) / mean_baseline_mae
)

print(
    "\nRandom Forest MAE improvement over the seven-day mean:",
    f"{improvement_percent:.2f}%"
)
print("Positive = improvement; negative = worse performance.")

In [ ]:
import matplotlib.pyplot as plt

validation_daily_totals = (
    validation_df
    .groupby("sale_date")[
        [
            TARGET_COLUMN,
            "pred_previous_7day_mean",
            "pred_random_forest",
        ]
    ]
    .sum()
)

plt.figure(figsize=(14, 5))

plt.plot(
    validation_daily_totals.index,
    validation_daily_totals[TARGET_COLUMN],
    label="Actual sales",
    color="black",
    linewidth=1.5
)

plt.plot(
    validation_daily_totals.index,
    validation_daily_totals["pred_previous_7day_mean"],
    label="Previous 7-day mean",
    alpha=0.8
)

plt.plot(
    validation_daily_totals.index,
    validation_daily_totals["pred_random_forest"],
    label="Random Forest v1",
    alpha=0.8
)

plt.title("Validation: total daily sales across selected products")
plt.xlabel("Target date")
plt.ylabel("Units")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    PROJECT_DIR / "reports" / "figures" / "validation_forecasts_v1.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import joblib

joblib.dump(
    rf_pipeline,
    PROJECT_DIR / "artifacts" / "random_forest_v1.joblib"
)

validation_comparison.to_csv(
    PROJECT_DIR / "reports" / "validation_model_comparison_v1.csv",
    index=False
)

validation_df[
    [
        "stock_code",
        "sale_date",
        TARGET_COLUMN,
        *prediction_columns.values(),
    ]
].to_csv(
    PROJECT_DIR / "reports" / "validation_predictions_v1.csv",
    index=False
)

print("Saved experimental model, comparison metrics, and validation predictions.")

In [ ]:
sales_pattern_summary = pd.DataFrame([
    {
        "split": split_name,
        "rows": len(frame),
        "zero_sales_percent": 100 * frame["daily_quantity"].eq(0).mean(),
        "mean_daily_units": frame["daily_quantity"].mean(),
        "median_daily_units": frame["daily_quantity"].median(),
    }
    for split_name, frame in [
        ("train", train_df),
        ("validation", validation_df),
    ]
])

display(sales_pattern_summary.round(3))

weekday_summary = (
    validation_df
    .groupby("day_of_week")
    .agg(
        product_days=("daily_quantity", "size"),
        mean_units=("daily_quantity", "mean"),
        median_units=("daily_quantity", "median"),
        zero_sales_fraction=(
            "daily_quantity",
            lambda values: values.eq(0).mean()
        ),
    )
)

weekday_summary["weekday"] = [
    "Monday", "Tuesday", "Wednesday", "Thursday",
    "Friday", "Saturday", "Sunday"
]

display(weekday_summary.round(3))

### Controlled revision: align the forest criterion with MAE

The first Random Forest used the default squared-error criterion.
It achieved the lowest validation RMSE but did not beat the always-zero reference on MAE.

We therefore test an absolute-error forest while keeping the feature set, training period,
validation period, and other hyperparameters unchanged.

This comparison investigates sensitivity to the training criterion. It does not establish
that either objective is optimal for inventory decisions, which may have asymmetric costs
for shortages and excess stock.

In [ ]:
from sklearn.base import clone

rf_mae_pipeline = clone(rf_pipeline)

rf_mae_pipeline.set_params(
    model__criterion="absolute_error"
)

print("V1 criterion:", rf_pipeline.get_params()["model__criterion"])
print("V2 criterion:", rf_mae_pipeline.get_params()["model__criterion"])

In [ ]:
training_start = perf_counter()

rf_mae_pipeline.fit(X_train, y_train)

mae_training_seconds = perf_counter() - training_start

validation_df["pred_random_forest_mae"] = (
    rf_mae_pipeline.predict(X_validation)
)

print(
    "Absolute-error forest training time:",
    f"{mae_training_seconds:.2f} seconds"
)

In [ ]:
median_feature_df = (
    daily_model_df
    .sort_values(["stock_code", "sale_date"])
    [["stock_code", "sale_date", "daily_quantity"]]
    .copy()
)

median_feature_df["pred_previous_7day_median"] = (
    median_feature_df
    .groupby("stock_code")["daily_quantity"]
    .transform(
        lambda series: (
            series.shift(1)
            .rolling(window=7, min_periods=7)
            .median()
        )
    )
)

validation_keys = pd.MultiIndex.from_frame(
    validation_df[["stock_code", "sale_date"]]
)

median_lookup = median_feature_df.set_index(
    ["stock_code", "sale_date"]
)["pred_previous_7day_median"]

validation_df["pred_previous_7day_median"] = (
    median_lookup.reindex(validation_keys).to_numpy()
)

assert validation_df["pred_previous_7day_median"].notna().all()

In [ ]:
validation_prediction_columns = {
    "Always zero reference": "pred_zero",
    "Previous-week same day": "pred_previous_week",
    "Previous 7-day mean": "pred_previous_7day_mean",
    "Previous 7-day median": "pred_previous_7day_median",
    "Random Forest v1: squared error": "pred_random_forest",
    "Random Forest v2: absolute error": "pred_random_forest_mae",
}

validation_comparison_v2 = pd.DataFrame([
    {
        "model": name,
        **regression_metrics(
            validation_df[TARGET_COLUMN],
            validation_df[column]
        )
    }
    for name, column in validation_prediction_columns.items()
]).sort_values("MAE").reset_index(drop=True)

display(validation_comparison_v2.round(3))

In [ ]:
joblib.dump(
    rf_mae_pipeline,
    PROJECT_DIR / "artifacts" / "random_forest_v2_absolute_error.joblib"
)

validation_comparison_v2.to_csv(
    PROJECT_DIR / "reports" / "validation_model_comparison_v2.csv",
    index=False
)

validation_df[
    [
        "stock_code",
        "sale_date",
        TARGET_COLUMN,
        *validation_prediction_columns.values(),
    ]
].to_csv(
    PROJECT_DIR / "reports" / "validation_predictions_v2.csv",
    index=False
)

print("Saved the revised model and validation comparison.")

## Model selection and final evaluation protocol

Random Forest v2, using the absolute-error criterion, is selected using validation MAE.

Validation results:
- MAE: 63.788 units per product-day.
- RMSE: 177.432 units.
- WAPE: 84.541%.
- Mean error: −26.713 units per product-day.

The model improves on the tested baselines but underpredicts sales in aggregate.

The previous-seven-day median is retained as the main simple forecasting comparator.
An always-zero reference is also reported.

For the final evaluation, we keep the fitted model unchanged:
- no refitting on validation data;
- no new hyperparameter tuning;
- no adjustment based on test results.

The test period covers 1 October–8 December 2011.
Evaluation follows the previously defined rolling one-day-ahead protocol.

In [ ]:
test_evaluation_df = (
    test_df
    .sort_values(["stock_code", "sale_date"])
    .copy()
)

assert train_df["sale_date"].max() < test_evaluation_df["sale_date"].min()
assert len(test_evaluation_df) == 1380

test_evaluation_df["pred_random_forest_v2"] = (
    rf_mae_pipeline.predict(
        test_evaluation_df[FEATURE_COLUMNS]
    )
)

test_evaluation_df["pred_zero"] = 0.0

# Reuse the previously constructed past-only median feature.
test_keys = pd.MultiIndex.from_frame(
    test_evaluation_df[["stock_code", "sale_date"]]
)

test_evaluation_df["pred_previous_7day_median"] = (
    median_lookup.reindex(test_keys).to_numpy()
)

prediction_columns_test = {
    "Random Forest v2: absolute error": "pred_random_forest_v2",
    "Previous 7-day median": "pred_previous_7day_median",
    "Always zero reference": "pred_zero",
}

assert test_evaluation_df[
    list(prediction_columns_test.values())
].notna().all().all()

display(
    test_evaluation_df[
        [
            "stock_code",
            "sale_date",
            TARGET_COLUMN,
            *prediction_columns_test.values(),
        ]
    ].head(10)
)

In [ ]:
test_results = pd.DataFrame([
    {
        "model": model_name,
        **regression_metrics(
            test_evaluation_df[TARGET_COLUMN],
            test_evaluation_df[prediction_column]
        )
    }
    for model_name, prediction_column in prediction_columns_test.items()
])

display(test_results.round(3))

In [ ]:
directional_results = []

for model_name, prediction_column in prediction_columns_test.items():
    actual = test_evaluation_df[TARGET_COLUMN].to_numpy(dtype=float)
    predicted = test_evaluation_df[prediction_column].to_numpy(dtype=float)

    underforecast = np.maximum(actual - predicted, 0)
    overforecast = np.maximum(predicted - actual, 0)

    directional_results.append({
        "model": model_name,
        "actual_units": actual.sum(),
        "forecast_units": predicted.sum(),
        "underforecast_units": underforecast.sum(),
        "overforecast_units": overforecast.sum(),
        "underforecast_product_days_percent": (
            100 * (predicted < actual).mean()
        ),
    })

directional_test_results = pd.DataFrame(directional_results)

display(directional_test_results.round(3))

In [ ]:
test_daily_totals = (
    test_evaluation_df
    .groupby("sale_date")[
        [
            TARGET_COLUMN,
            "pred_random_forest_v2",
            "pred_previous_7day_median",
        ]
    ]
    .sum()
)

plt.figure(figsize=(14, 5))

plt.plot(
    test_daily_totals.index,
    test_daily_totals[TARGET_COLUMN],
    label="Actual recorded sales",
    color="black",
    linewidth=1.5
)

plt.plot(
    test_daily_totals.index,
    test_daily_totals["pred_random_forest_v2"],
    label="Random Forest v2",
    alpha=0.85
)

plt.plot(
    test_daily_totals.index,
    test_daily_totals["pred_previous_7day_median"],
    label="Previous 7-day median",
    alpha=0.75
)

plt.title("Held-out test: total daily sales across selected products")
plt.xlabel("Target date")
plt.ylabel("Units")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    PROJECT_DIR / "reports" / "figures" / "test_forecasts.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()

In [ ]:
test_results.to_csv(
    PROJECT_DIR / "reports" / "final_test_metrics.csv",
    index=False
)

directional_test_results.to_csv(
    PROJECT_DIR / "reports" / "test_directional_errors.csv",
    index=False
)

test_evaluation_df[
    [
        "stock_code",
        "sale_date",
        TARGET_COLUMN,
        *prediction_columns_test.values(),
    ]
].to_csv(
    PROJECT_DIR / "reports" / "final_test_predictions.csv",
    index=False
)

joblib.dump(
    rf_mae_pipeline,
    PROJECT_DIR / "artifacts" / "selected_forecasting_pipeline.joblib"
)

print("Saved selected model and final test evaluation artifacts.")

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import shutil
from pathlib import Path

DRIVE_PROJECT_DIR = (
    Path("/content/drive/MyDrive")
    / "Colab Notebooks"
    / "retail-demand-simulator"
)

assert DRIVE_PROJECT_DIR.exists(), (
    "Project folder not found. Check its location and spelling in Google Drive."
)

BACKUP_DIR = DRIVE_PROJECT_DIR / "project_files"
BACKUP_DIR.mkdir(parents=True, exist_ok=True)

for relative_folder in [
    "data/processed",
    "artifacts",
    "reports",
]:
    source = PROJECT_DIR / relative_folder
    destination = BACKUP_DIR / relative_folder

    shutil.copytree(
        source,
        destination,
        dirs_exist_ok=True
    )

print("Copied processed data, model artifacts, and reports to Google Drive.")
print("Backup folder:", BACKUP_DIR)

## One-day resource-allocation what-if simulator

The simulator uses saved historical one-day-ahead forecasts.

Users specify:
- a historical forecast date;
- an assumed percentage change in demand;
- starting stock per product;
- replenishment per product.

Available stock and replenishment are hypothetical inputs because the dataset does not
contain historical inventory records.

Projected shortage is the positive difference between scenario demand and available units.
It is not an observed historical stockout.

The demand-change control is a deterministic scenario assumption, not a causal estimate
or a calibrated uncertainty interval.

Replenishment is assumed to be available before demand is served.

In [ ]:
def simulate_one_day(
    forecast_table,
    scenario_date,
    demand_change_percent=0.0,
    starting_stock_units=50,
    replenishment_units=0,
):
    """
    Simulate one day using the same stock assumptions for every product.
    Actual sales are deliberately excluded from the calculation.
    """
    if not np.isfinite(demand_change_percent):
        raise ValueError("Demand change must be finite.")

    if demand_change_percent < -100:
        raise ValueError("Demand change cannot be below -100%.")

    for name, value in [
        ("starting_stock_units", starting_stock_units),
        ("replenishment_units", replenishment_units),
    ]:
        if not np.isfinite(value) or value < 0:
            raise ValueError(f"{name} must be finite and non-negative.")

    selected_date = pd.Timestamp(scenario_date).normalize()

    result = forecast_table.loc[
        forecast_table["sale_date"].eq(selected_date),
        ["stock_code", "sale_date", "pred_random_forest_v2"]
    ].copy()

    if result.empty:
        raise ValueError("No forecasts are available for this date.")

    if result["stock_code"].duplicated().any():
        raise ValueError("Expected one forecast per product.")

    result = result.rename(
        columns={"pred_random_forest_v2": "baseline_forecast"}
    )

    assert result["baseline_forecast"].notna().all()
    assert result["baseline_forecast"].ge(0).all()

    result["scenario_demand"] = (
        result["baseline_forecast"]
        * (1 + demand_change_percent / 100)
    )

    result["starting_stock"] = starting_stock_units
    result["replenishment"] = replenishment_units

    result["available_units"] = (
        result["starting_stock"] + result["replenishment"]
    )

    result["projected_shortage"] = (
        result["scenario_demand"] - result["available_units"]
    ).clip(lower=0)

    result["projected_surplus"] = (
        result["available_units"] - result["scenario_demand"]
    ).clip(lower=0)

    return result.sort_values(
        ["projected_shortage", "stock_code"],
        ascending=[False, True]
    ).reset_index(drop=True)

In [ ]:
SCENARIO_DATE = "2011-11-01"
DEMAND_CHANGE_PERCENT = 20
STARTING_STOCK_UNITS = 50
REPLENISHMENT_UNITS = 10

scenario_df = simulate_one_day(
    forecast_table=test_evaluation_df,
    scenario_date=SCENARIO_DATE,
    demand_change_percent=DEMAND_CHANGE_PERCENT,
    starting_stock_units=STARTING_STOCK_UNITS,
    replenishment_units=REPLENISHMENT_UNITS,
)

scenario_summary = pd.Series({
    "baseline_forecast_units": scenario_df["baseline_forecast"].sum(),
    "scenario_demand_units": scenario_df["scenario_demand"].sum(),
    "available_units": scenario_df["available_units"].sum(),
    "projected_shortage_units": scenario_df["projected_shortage"].sum(),
    "projected_surplus_units": scenario_df["projected_surplus"].sum(),
    "products_with_projected_shortage": int(
        scenario_df["projected_shortage"].gt(0).sum()
    ),
})

display(scenario_summary.round(2))
display(scenario_df.round(2))

In [ ]:
plt.figure(figsize=(12, 6))

plt.barh(
    scenario_df["stock_code"].astype(str),
    scenario_df["projected_shortage"],
    color="tomato"
)

plt.gca().invert_yaxis()

plt.title(
    f"Projected product shortages: {SCENARIO_DATE}\n"
    f"Demand change: {DEMAND_CHANGE_PERCENT:+.0f}% | "
    f"Stock/product: {STARTING_STOCK_UNITS} | "
    f"Replenishment/product: {REPLENISHMENT_UNITS}"
)

plt.xlabel("Projected shortage units")
plt.ylabel("Product code")
plt.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
baseline_scenario = simulate_one_day(
    test_evaluation_df, SCENARIO_DATE, 0, 50, 10
).set_index("stock_code").sort_index()

increased_demand_scenario = simulate_one_day(
    test_evaluation_df, SCENARIO_DATE, 20, 50, 10
).set_index("stock_code").sort_index()

more_stock_scenario = simulate_one_day(
    test_evaluation_df, SCENARIO_DATE, 0, 100, 10
).set_index("stock_code").sort_index()

assert np.allclose(
    baseline_scenario["scenario_demand"],
    baseline_scenario["baseline_forecast"]
)

assert (
    increased_demand_scenario["projected_shortage"]
    >= baseline_scenario["projected_shortage"] - 1e-9
).all()

assert (
    more_stock_scenario["projected_shortage"]
    <= baseline_scenario["projected_shortage"] + 1e-9
).all()

print("Passed: zero-change, demand-increase, and stock-increase checks.")

In [ ]:
PUBLICATION_DIR = PROJECT_DIR / "publication"
PUBLICATION_DATA_DIR = PUBLICATION_DIR / "data"

PUBLICATION_DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Dashboard files will be created in:", PUBLICATION_DIR)

In [ ]:
dashboard_forecasts = (
    test_evaluation_df[
        ["stock_code", "sale_date", "pred_random_forest_v2"]
    ]
    .rename(
        columns={"pred_random_forest_v2": "baseline_forecast"}
    )
    .sort_values(["sale_date", "stock_code"])
    .reset_index(drop=True)
)

assert dashboard_forecasts.shape[0] == 1380
assert not dashboard_forecasts.duplicated(
    ["stock_code", "sale_date"]
).any()

assert dashboard_forecasts["baseline_forecast"].notna().all()
assert dashboard_forecasts["baseline_forecast"].ge(0).all()

DASHBOARD_DATA_PATH = (
    PUBLICATION_DATA_DIR / "historical_forecasts.csv"
)

dashboard_forecasts.to_csv(DASHBOARD_DATA_PATH, index=False)

print("Exported rows:", len(dashboard_forecasts))
print("Columns:", dashboard_forecasts.columns.tolist())
display(dashboard_forecasts.head())

In [ ]:
%pip install -q streamlit

In [ ]:
app_code = '''
from pathlib import Path

import pandas as pd
import streamlit as st

st.set_page_config(
    page_title="Retail Demand Simulator",
    page_icon="📦",
    layout="wide",
)

DATA_PATH = (
    Path(__file__).resolve().parent
    / "data"
    / "historical_forecasts.csv"
)


@st.cache_data
def load_forecasts():
    data = pd.read_csv(
        DATA_PATH,
        dtype={"stock_code": "string"},
        parse_dates=["sale_date"],
    )

    required = {"stock_code", "sale_date", "baseline_forecast"}

    if not required.issubset(data.columns):
        raise ValueError("Forecast file is missing required columns.")

    if data[list(required)].isna().any().any():
        raise ValueError("Forecast file contains missing values.")

    if data.duplicated(["stock_code", "sale_date"]).any():
        raise ValueError("Forecast file contains duplicate product-date rows.")

    if not data["baseline_forecast"].ge(0).all():
        raise ValueError("Forecast quantities must be non-negative.")

    return data.sort_values(["sale_date", "stock_code"])


def simulate(data, selected_date, demand_change, stock, replenishment):
    result = data.loc[
        data["sale_date"].dt.date.eq(selected_date)
    ].copy()

    result["scenario_demand"] = (
        result["baseline_forecast"] * (1 + demand_change / 100)
    )

    result["starting_stock"] = stock
    result["replenishment"] = replenishment
    result["available_units"] = stock + replenishment

    result["projected_shortage"] = (
        result["scenario_demand"] - result["available_units"]
    ).clip(lower=0)

    result["projected_surplus"] = (
        result["available_units"] - result["scenario_demand"]
    ).clip(lower=0)

    return result.sort_values(
        ["projected_shortage", "stock_code"],
        ascending=[False, True],
    )


st.title("Retail Demand & Resource Allocation Simulator")

st.caption(
    "Historical scenario explorer | "
    "20 products | One-day-ahead Random Forest forecasts"
)

st.warning(
    "This is a research demonstration, not a live inventory system. "
    "Stock and replenishment are hypothetical inputs. "
    "The model underforecast total sales in held-out testing; "
    "scenario shortages are not observed stockouts or shortage probabilities."
)

forecasts = load_forecasts()
available_dates = sorted(forecasts["sale_date"].dt.date.unique())

preferred_date = pd.Timestamp("2011-11-01").date()
default_index = (
    available_dates.index(preferred_date)
    if preferred_date in available_dates
    else 0
)

with st.sidebar:
    st.header("Scenario assumptions")

    selected_date = st.selectbox(
        "Historical forecast date",
        options=available_dates,
        index=default_index,
        format_func=lambda value: value.strftime("%d %B %Y"),
    )

    demand_change = st.slider(
        "Assumed demand change (%)",
        min_value=-100,
        max_value=100,
        value=20,
        step=5,
    )

    stock = st.number_input(
        "Starting stock per product (units)",
        min_value=0,
        value=50,
        step=10,
    )

    replenishment = st.number_input(
        "Replenishment per product (units)",
        min_value=0,
        value=10,
        step=10,
    )

    st.caption(
        "The same stock settings apply to all products. "
        "Replenishment is assumed available before demand is served."
    )

scenario = simulate(
    forecasts,
    selected_date,
    demand_change,
    stock,
    replenishment,
)

st.subheader(
    f"Scenario for {selected_date.strftime('%d %B %Y')}"
)

c1, c2, c3, c4 = st.columns(4)

c1.metric(
    "Scenario demand",
    f"{scenario['scenario_demand'].sum():,.1f} units",
)

c2.metric(
    "Available stock",
    f"{scenario['available_units'].sum():,.0f} units",
)

c3.metric(
    "Projected shortage",
    f"{scenario['projected_shortage'].sum():,.1f} units",
)

c4.metric(
    "Products with shortages",
    f"{int(scenario['projected_shortage'].gt(0).sum())}"
    f" / {len(scenario)}",
)

st.caption(
    f"Baseline forecast: {scenario['baseline_forecast'].sum():,.1f} units. "
    f"Projected surplus: {scenario['projected_surplus'].sum():,.1f} units. "
    "Shortages and surpluses are calculated separately for each product."
)

st.subheader("Projected shortage by product")

shortage_chart = scenario[
    ["stock_code", "projected_shortage"]
].set_index("stock_code")

st.bar_chart(shortage_chart)

st.subheader("Forecast, scenario demand, and available stock")

comparison_chart = scenario[
    [
        "stock_code",
        "baseline_forecast",
        "scenario_demand",
        "available_units",
    ]
].set_index("stock_code")

st.bar_chart(comparison_chart)

st.subheader("Product-level scenario results")

st.dataframe(
    scenario.drop(columns=["sale_date"]).round(2),
    hide_index=True,
    width="stretch",
)

export = scenario.copy()
export["demand_change_percent"] = demand_change

st.download_button(
    label="Download scenario as CSV",
    data=export.to_csv(index=False).encode("utf-8"),
    file_name=f"retail_scenario_{selected_date.isoformat()}.csv",
    mime="text/csv",
)

with st.expander("Methodology and limitations"):
    st.markdown("""
- Forecasts were generated using a Random Forest with an absolute-error criterion.
- Products were selected using training-period data only.
- Evaluation used rolling one-day-ahead predictions.
- Held-out test MAE: 65.535 units per product-day.
- Held-out test WAPE: 75.706%.
- Total test sales: 119,459 units; total forecasts: approximately 76,075 units.
- Recorded sales are a proxy for demand.
- Inventory and replenishment are user assumptions, not historical observations.
- Demand adjustments are deterministic scenarios, not causal estimates.
- This app does not optimize replenishment, model delivery delays, or estimate shortage probabilities.
- Fractional quantities are retained for analytical display; they are not rounded purchase recommendations.
""")

st.caption(
    "Data provenance: UCI Online Retail II, Chen (2012), "
    "DOI: 10.24432/C5CG6D. "
    "Dashboard data contains derived forecasts only."
)
'''

APP_PATH = PUBLICATION_DIR / "app.py"
APP_PATH.write_text(app_code, encoding="utf-8")

print("Created:", APP_PATH)

In [ ]:
from importlib.metadata import version

dashboard_requirements = (
    f"streamlit=={version('streamlit')}\n"
    f"pandas=={version('pandas')}\n"
)

requirements_path = PUBLICATION_DIR / "requirements.txt"
requirements_path.write_text(
    dashboard_requirements,
    encoding="utf-8"
)

print(requirements_path.read_text())

In [ ]:
from streamlit.testing.v1 import AppTest

app_test = AppTest.from_file(str(APP_PATH))
app_test.run(timeout=60)

assert len(app_test.exception) == 0, [
    exception.message for exception in app_test.exception
]

print("Application loaded without exceptions.")

for metric in app_test.metric:
    print(f"{metric.label}: {metric.value}")

app_test.slider[0].set_value(40)
app_test.run(timeout=60)

assert len(app_test.exception) == 0, [
    exception.message for exception in app_test.exception
]

print("\nApplication reran after changing demand to +40%.")
for metric in app_test.metric:
    print(f"{metric.label}: {metric.value}")

In [ ]:
DRIVE_PUBLICATION_DIR = DRIVE_PROJECT_DIR / "publication"

shutil.copytree(
    PUBLICATION_DIR,
    DRIVE_PUBLICATION_DIR,
    dirs_exist_ok=True,
)

print("Saved dashboard publication folder to Drive:")
print(DRIVE_PUBLICATION_DIR)

for path in sorted(PUBLICATION_DIR.rglob("*")):
    if path.is_file():
        print(
            path.relative_to(PUBLICATION_DIR),
            f"({path.stat().st_size:,} bytes)"
        )

## Final research conclusion

This project investigated whether historical retail transactions could support
one-day-ahead product-sales forecasting and stock-shortage scenario analysis.

The pipeline audited and cleaned 1,067,371 transaction records, selected 20 products
using training-period information, and constructed daily sales histories.

Investigation of an unusually large sales spike identified a 74,215-unit order
reversed 16 minutes later. The positive order was excluded through a documented,
case-specific rule rather than an arbitrary outlier threshold.

A Random Forest using an absolute-error criterion was selected on validation MAE.
On the held-out test period, it achieved:
- MAE: 65.535 units per product-day.
- RMSE: 217.652 units.
- WAPE: 75.706%.

Its test MAE was approximately 11.8% lower than the previous-seven-day median baseline.
However, total forecasts were approximately 76,075 units against 119,459 recorded
sales, indicating substantial aggregate underprediction.

The deployed Streamlit application uses precomputed historical forecasts to explore
deterministic demand and stock scenarios. It does not provide live forecasts,
estimate stockout probabilities, or optimize replenishment.

The project demonstrates how a forecasting model can support exploratory decisions
while also showing why prediction error, data quality, and operational assumptions
must remain visible to users.

## Automated dashboard tests

The dashboard is tested using Streamlit AppTest and pytest.

The tests cover the forecast input schema, scenario arithmetic, application execution,
and expected responses to changes in demand and stock.

These checks validate application behavior, not forecasting accuracy.
Forecast accuracy is assessed separately using the chronological evaluation.

In [ ]:
TESTS_DIR = PUBLICATION_DIR / "tests"
TESTS_DIR.mkdir(parents=True, exist_ok=True)

assert (PUBLICATION_DIR / "app.py").exists()
assert (PUBLICATION_DIR / "data" / "historical_forecasts.csv").exists()

print("Test folder:", TESTS_DIR)

In [ ]:
%pip install -q pytest

In [ ]:
test_code = '''
from pathlib import Path

import pandas as pd
import pytest
from streamlit.testing.v1 import AppTest

ROOT = Path(__file__).resolve().parents[1]
APP_PATH = ROOT / "app.py"
DATA_PATH = ROOT / "data" / "historical_forecasts.csv"


def load_data():
    return pd.read_csv(
        DATA_PATH,
        dtype={"stock_code": "string"},
        parse_dates=["sale_date"],
    )


def start_app():
    app = AppTest.from_file(str(APP_PATH))
    app.run(timeout=60)

    assert not app.exception, [
        error.message for error in app.exception
    ]

    return app


def displayed_metrics(app):
    return {
        metric.label: metric.value
        for metric in app.metric
    }


def metric_number(app, label):
    value = displayed_metrics(app)[label]
    return float(value.replace(" units", "").replace(",", ""))


def test_forecast_data_contract():
    data = load_data()

    assert set(data.columns) == {
        "stock_code",
        "sale_date",
        "baseline_forecast",
    }

    assert not data.isna().any().any()

    assert not data.duplicated(
        ["stock_code", "sale_date"]
    ).any()

    assert data["baseline_forecast"].ge(0).all()

    assert data["stock_code"].nunique() == 20

    assert data.groupby("sale_date").size().eq(20).all()

    assert data["sale_date"].min() == pd.Timestamp("2011-10-01")
    assert data["sale_date"].max() == pd.Timestamp("2011-12-08")


def test_app_loads():
    app = start_app()

    assert len(app.selectbox) == 1
    assert len(app.slider) == 1
    assert len(app.number_input) == 2

    assert set(displayed_metrics(app)) == {
        "Scenario demand",
        "Available stock",
        "Projected shortage",
        "Products with shortages",
    }


@pytest.mark.parametrize("change", [-100, 0, 20, 40])
def test_displayed_scenario_arithmetic(change):
    app = start_app()

    app.slider[0].set_value(change)
    app.number_input[0].set_value(50)
    app.number_input[1].set_value(10)
    app.run(timeout=60)

    assert not app.exception

    data = load_data()
    selected_date = app.selectbox[0].value

    forecasts = data.loc[
        data["sale_date"].dt.date.eq(selected_date),
        "baseline_forecast",
    ]

    scenario_demand = forecasts * (1 + change / 100)
    available_per_product = 60

    expected_demand = scenario_demand.sum()

    expected_shortage = (
        scenario_demand - available_per_product
    ).clip(lower=0).sum()

    expected_affected = int(
        scenario_demand.gt(available_per_product).sum()
    )

    metrics = displayed_metrics(app)

    assert metrics["Scenario demand"] == (
        f"{expected_demand:,.1f} units"
    )

    assert metrics["Available stock"] == (
        f"{available_per_product * len(forecasts):,.0f} units"
    )

    assert metrics["Projected shortage"] == (
        f"{expected_shortage:,.1f} units"
    )

    assert metrics["Products with shortages"] == (
        f"{expected_affected} / {len(forecasts)}"
    )


def test_more_demand_does_not_reduce_shortage():
    app = start_app()

    app.slider[0].set_value(0)
    app.run(timeout=60)
    assert not app.exception

    baseline_shortage = metric_number(
        app, "Projected shortage"
    )

    app.slider[0].set_value(40)
    app.run(timeout=60)
    assert not app.exception

    increased_shortage = metric_number(
        app, "Projected shortage"
    )

    assert increased_shortage >= baseline_shortage


def test_more_stock_does_not_increase_shortage():
    app = start_app()

    app.number_input[0].set_value(50)
    app.run(timeout=60)
    assert not app.exception

    original_shortage = metric_number(
        app, "Projected shortage"
    )

    app.number_input[0].set_value(100)
    app.run(timeout=60)
    assert not app.exception

    revised_shortage = metric_number(
        app, "Projected shortage"
    )

    assert revised_shortage <= original_shortage
'''

TEST_PATH = TESTS_DIR / "test_dashboard.py"
TEST_PATH.write_text(test_code, encoding="utf-8")

print("Created:", TEST_PATH)

In [ ]:
import subprocess
import sys

test_result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "tests/test_dashboard.py",
        "-q",
    ],
    cwd=PUBLICATION_DIR,
    capture_output=True,
    text=True,
)

print(test_result.stdout)

if test_result.stderr:
    print(test_result.stderr)

assert test_result.returncode == 0, (
    "Some tests failed. Inspect the output before publishing."
)

In [ ]:
from importlib.metadata import version

dev_requirements = (
    "-r requirements.txt\n"
    f"pytest=={version('pytest')}\n"
)

DEV_REQUIREMENTS_PATH = (
    PUBLICATION_DIR / "requirements-dev.txt"
)

DEV_REQUIREMENTS_PATH.write_text(
    dev_requirements,
    encoding="utf-8"
)

print(DEV_REQUIREMENTS_PATH.read_text())

In [ ]:
shutil.copytree(
    PUBLICATION_DIR,
    DRIVE_PUBLICATION_DIR,
    dirs_exist_ok=True,
    ignore=shutil.ignore_patterns(
        "__pycache__",
        ".pytest_cache",
        "*.pyc",
    ),
)

print("Saved test files and development requirements to Drive.")

In [ ]:
print(
    (PUBLICATION_DIR / "requirements-dev.txt")
    .read_text(encoding="utf-8")
)

In [ ]:
print(
    (PUBLICATION_DIR / "tests" / "test_dashboard.py")
    .read_text(encoding="utf-8")
)

## Standalone pipeline reproducibility check

The following script reproduces the selected cleaning, feature-engineering,
training, and evaluation workflow in a separate Python process.

It does not depend on notebook variables or retune the selected model.
Its outputs are compared with the original notebook results.

In [ ]:
training_script = r'''
import argparse
import hashlib
import json
import platform
from importlib.metadata import version
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder


TRAIN_END = pd.Timestamp("2011-07-31")
VALIDATION_END = pd.Timestamp("2011-09-30")
MODEL_END = pd.Timestamp("2011-12-08")

COLUMN_MAPPING = {
    "Invoice": "invoice_no",
    "StockCode": "stock_code",
    "Description": "description",
    "Quantity": "quantity",
    "InvoiceDate": "invoice_date",
    "Price": "unit_price",
    "Customer ID": "customer_id",
    "Country": "country",
}

NUMERIC_FEATURES = [
    "lag_1",
    "lag_7",
    "lag_14",
    "rolling_mean_7",
    "rolling_mean_28",
    "day_of_week",
    "month",
]

FEATURES = ["stock_code"] + NUMERIC_FEATURES


def load_and_clean(workbook):
    frames = []

    with pd.ExcelFile(workbook) as excel:
        for sheet in excel.sheet_names:
            frame = pd.read_excel(excel, sheet_name=sheet)
            frame = frame.rename(columns=COLUMN_MAPPING)

            missing = set(COLUMN_MAPPING.values()) - set(frame.columns)
            if missing:
                raise ValueError(f"Missing columns in {sheet}: {missing}")

            frame["source_sheet"] = sheet
            frames.append(frame)

    raw = pd.concat(frames, ignore_index=True)

    for column in ["invoice_no", "stock_code", "description", "country"]:
        raw[column] = raw[column].astype("string")

    raw["invoice_date"] = pd.to_datetime(
        raw["invoice_date"], errors="coerce"
    )

    for column in ["quantity", "unit_price", "customer_id"]:
        raw[column] = pd.to_numeric(raw[column], errors="coerce")

    if raw[
        ["invoice_no", "stock_code", "invoice_date", "quantity", "unit_price"]
    ].isna().any().any():
        raise ValueError("Missing essential fields; inspect before continuing.")

    positive_pair = (
        raw["invoice_no"].eq("541431")
        & raw["stock_code"].eq("23166")
        & raw["customer_id"].eq(12346)
        & raw["quantity"].eq(74215)
        & raw["invoice_date"].eq(pd.Timestamp("2011-01-18 10:01:00"))
    )

    negative_pair = (
        raw["invoice_no"].eq("C541433")
        & raw["stock_code"].eq("23166")
        & raw["customer_id"].eq(12346)
        & raw["quantity"].eq(-74215)
        & raw["invoice_date"].eq(pd.Timestamp("2011-01-18 10:17:00"))
    )

    if positive_pair.sum() != 1 or negative_pair.sum() != 1:
        raise ValueError("The documented reversal pair was not uniquely found.")

    if (
        raw.loc[positive_pair, "unit_price"].iloc[0]
        != raw.loc[negative_pair, "unit_price"].iloc[0]
    ):
        raise ValueError("Reversal pair prices do not match.")

    audit = [{"step": "raw", "rows": len(raw)}]

    sales = raw.drop_duplicates().copy()
    audit.append({"step": "deduplicated", "rows": len(sales)})

    sales = sales.loc[
        ~sales["invoice_no"].str.upper().str.startswith("C")
    ].copy()
    audit.append({"step": "exclude_cancellations", "rows": len(sales)})

    sales = sales.loc[sales["quantity"].ge(0)].copy()
    audit.append({"step": "exclude_negative_quantities", "rows": len(sales)})

    sales = sales.loc[sales["unit_price"].gt(0)].copy()
    audit.append({"step": "exclude_nonpositive_prices", "rows": len(sales)})

    sales = sales.loc[~sales.index.isin(raw.index[positive_pair])].copy()
    audit.append({"step": "exclude_verified_reversal", "rows": len(sales)})

    sales["sale_date"] = sales["invoice_date"].dt.normalize()

    sales = sales.loc[
        sales["sale_date"].between("2009-12-01", MODEL_END)
    ].copy()

    audit.append({"step": "modeling_date_window", "rows": len(sales)})

    return sales, pd.DataFrame(audit)


def build_features(sales):
    training_sales = sales.loc[sales["sale_date"].le(TRAIN_END)]

    summary = (
        training_sales.groupby("stock_code")
        .agg(
            active_sales_days=("sale_date", "nunique"),
            first_observed_sale=("sale_date", "min"),
            last_training_sale=("sale_date", "max"),
            training_units=("quantity", "sum"),
        )
        .reset_index()
    )

    summary["calendar_span_days"] = (
        summary["last_training_sale"] - summary["first_observed_sale"]
    ).dt.days + 1

    selected = (
        summary.loc[
            summary["active_sales_days"].ge(30)
            & summary["calendar_span_days"].ge(180)
            & summary["training_units"].ge(100)
        ]
        .sort_values(
            ["training_units", "stock_code"],
            ascending=[False, True],
        )
        .head(20)
        .copy()
    )

    if len(selected) != 20:
        raise ValueError("Expected 20 eligible products.")

    observed = (
        sales.loc[sales["stock_code"].isin(selected["stock_code"])]
        .groupby(["stock_code", "sale_date"], as_index=False)
        .agg(daily_quantity=("quantity", "sum"))
    )

    calendars = [
        pd.DataFrame({
            "stock_code": row.stock_code,
            "sale_date": pd.date_range(row.first_observed_sale, MODEL_END),
        })
        for row in selected.itertuples(index=False)
    ]

    daily = pd.concat(calendars, ignore_index=True).merge(
        observed,
        on=["stock_code", "sale_date"],
        how="left",
        validate="one_to_one",
    )

    daily["daily_quantity"] = daily["daily_quantity"].fillna(0).astype("int64")
    daily = daily.sort_values(["stock_code", "sale_date"]).reset_index(drop=True)

    for lag in [1, 7, 14]:
        daily[f"lag_{lag}"] = (
            daily.groupby("stock_code")["daily_quantity"].shift(lag)
        )

    for window in [7, 28]:
        daily[f"rolling_mean_{window}"] = (
            daily.groupby("stock_code")["daily_quantity"]
            .transform(
                lambda values: values.shift(1)
                .rolling(window, min_periods=window)
                .mean()
            )
        )

    daily["previous_7day_median"] = (
        daily.groupby("stock_code")["daily_quantity"]
        .transform(
            lambda values: values.shift(1)
            .rolling(7, min_periods=7)
            .median()
        )
    )

    daily["day_of_week"] = daily["sale_date"].dt.dayofweek
    daily["month"] = daily["sale_date"].dt.month

    daily["split"] = "test"
    daily.loc[daily["sale_date"].le(VALIDATION_END), "split"] = "validation"
    daily.loc[daily["sale_date"].le(TRAIN_END), "split"] = "train"

    return daily.dropna(subset=NUMERIC_FEATURES).copy(), selected


def make_model():
    preprocessing = ColumnTransformer([
        (
            "product",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            ["stock_code"],
        ),
        ("numeric", "passthrough", NUMERIC_FEATURES),
    ])

    return Pipeline([
        ("preprocessor", preprocessing),
        (
            "model",
            RandomForestRegressor(
                n_estimators=200,
                criterion="absolute_error",
                max_depth=12,
                min_samples_leaf=5,
                max_features=1.0,
                random_state=42,
                n_jobs=2,
            ),
        ),
    ])


def metrics(actual, predicted):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)

    return {
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
        "WAPE_percent": (
            100 * np.abs(actual - predicted).sum() / actual.sum()
            if actual.sum() > 0 else np.nan
        ),
        "mean_error_pred_minus_actual": (predicted - actual).mean(),
    }


def main():
    parser = argparse.ArgumentParser(
        description="Reproduce the selected retail forecasting pipeline."
    )
    parser.add_argument("--input", type=Path, required=True)
    parser.add_argument("--output", type=Path, required=True)
    args = parser.parse_args()

    if not args.input.is_file():
        raise FileNotFoundError(args.input)

    args.output.mkdir(parents=True, exist_ok=True)

    sales, audit = load_and_clean(args.input)
    features, selected = build_features(sales)

    train = features.loc[features["split"].eq("train")]
    model = make_model()
    model.fit(train[FEATURES], train["daily_quantity"])

    results = []

    for split in ["validation", "test"]:
        evaluation = features.loc[features["split"].eq(split)].copy()
        evaluation["baseline_forecast"] = model.predict(evaluation[FEATURES])
        evaluation["zero_reference"] = 0.0

        for name, column in [
            ("Random Forest v2", "baseline_forecast"),
            ("Previous 7-day median", "previous_7day_median"),
            ("Always zero", "zero_reference"),
        ]:
            results.append({
                "split": split,
                "model": name,
                **metrics(evaluation["daily_quantity"], evaluation[column]),
            })

        evaluation[
            [
                "stock_code", "sale_date", "daily_quantity",
                "baseline_forecast", "previous_7day_median",
            ]
        ].to_csv(args.output / f"{split}_predictions.csv", index=False)

        if split == "test":
            evaluation[
                ["stock_code", "sale_date", "baseline_forecast"]
            ].to_csv(args.output / "historical_forecasts.csv", index=False)

    result_table = pd.DataFrame(results)
    result_table.to_csv(args.output / "metrics.csv", index=False)
    audit.to_csv(args.output / "cleaning_summary.csv", index=False)
    selected.to_csv(args.output / "selected_products.csv", index=False)

    features.groupby("split").size().to_csv(
        args.output / "split_row_counts.csv",
        header=["rows"],
    )

    joblib.dump(model, args.output / "selected_model.joblib")

    metadata = {
        "python": platform.python_version(),
        "packages": {
            package: version(package)
            for package in [
                "pandas", "numpy", "scikit-learn",
                "scipy", "openpyxl", "joblib",
            ]
        },
        "input_sha256": hashlib.sha256(args.input.read_bytes()).hexdigest(),
        "features": FEATURES,
        "model_parameters": model.named_steps["model"].get_params(),
        "training_end": str(TRAIN_END.date()),
        "validation_end": str(VALIDATION_END.date()),
        "test_end": str(MODEL_END.date()),
        "refitted_on_validation": False,
    }

    (args.output / "run_metadata.json").write_text(
        json.dumps(metadata, indent=2),
        encoding="utf-8",
    )

    print(result_table.round(3).to_string(index=False))
    print("\nOutputs saved to:", args.output)


if __name__ == "__main__":
    main()
'''

TRAIN_SCRIPT_PATH = PUBLICATION_DIR / "train.py"
TRAIN_SCRIPT_PATH.write_text(training_script, encoding="utf-8")

print("Created:", TRAIN_SCRIPT_PATH)

In [ ]:
from importlib.metadata import version

training_packages = [
    "pandas",
    "numpy",
    "scikit-learn",
    "scipy",
    "openpyxl",
    "joblib",
]

training_requirements = "\n".join(
    f"{package}=={version(package)}"
    for package in training_packages
) + "\n"

(PUBLICATION_DIR / "requirements-training.txt").write_text(
    training_requirements,
    encoding="utf-8",
)

print(training_requirements)

In [ ]:
import subprocess
import sys

WORKBOOK_PATH = (
    PROJECT_DIR / "data" / "raw" / "online_retail_II.xlsx"
)

REPRODUCTION_DIR = PROJECT_DIR / "reproduction_run"

assert WORKBOOK_PATH.exists(), (
    "The original workbook is missing. Rerun its download and extraction cells."
)

run_result = subprocess.run(
    [
        sys.executable,
        str(TRAIN_SCRIPT_PATH),
        "--input",
        str(WORKBOOK_PATH),
        "--output",
        str(REPRODUCTION_DIR),
    ],
    capture_output=True,
    text=True,
)

print(run_result.stdout)

if run_result.stderr:
    print(run_result.stderr)

assert run_result.returncode == 0, (
    "Standalone execution failed. Inspect the error before continuing."
)

In [ ]:
original_forecasts = pd.read_csv(
    PUBLICATION_DIR / "data" / "historical_forecasts.csv",
    dtype={"stock_code": "string"},
    parse_dates=["sale_date"],
)

reproduced_forecasts = pd.read_csv(
    REPRODUCTION_DIR / "historical_forecasts.csv",
    dtype={"stock_code": "string"},
    parse_dates=["sale_date"],
)

comparison = original_forecasts.merge(
    reproduced_forecasts,
    on=["stock_code", "sale_date"],
    how="outer",
    suffixes=("_original", "_reproduced"),
    indicator=True,
    validate="one_to_one",
)

assert comparison["_merge"].eq("both").all(), (
    "The original and reproduced product-date keys differ."
)

differences = (
    comparison["baseline_forecast_original"]
    - comparison["baseline_forecast_reproduced"]
).abs()

print("Compared product-date rows:", len(comparison))
print("Largest absolute prediction difference:", differences.max())

np.testing.assert_allclose(
    comparison["baseline_forecast_original"],
    comparison["baseline_forecast_reproduced"],
    rtol=1e-9,
    atol=1e-9,
)

print("Passed: standalone forecasts match the existing forecasts.")

In [ ]:
shutil.copytree(
    PUBLICATION_DIR,
    DRIVE_PUBLICATION_DIR,
    dirs_exist_ok=True,
    ignore=shutil.ignore_patterns(
        "__pycache__",
        ".pytest_cache",
        "*.pyc",
    ),
)

shutil.copytree(
    REPRODUCTION_DIR,
    DRIVE_PROJECT_DIR / "project_files" / "reproduction_run",
    dirs_exist_ok=True,
)

print("Saved training source files and reproduction outputs to Drive.")

## Standalone reproduction result

The selected workflow was extracted into `train.py` and executed in a separate
Python process using the original workbook.

The script reproduced all 1,380 historical test forecasts with a maximum absolute
difference of 0.0 against the notebook-generated forecast file.

Validation and test metrics also matched the original results.

This confirms implementation equivalence in the recorded development environment.
No model parameters were changed based on the reproduction run.